# 过拟合、正则化与早停

## 学习目标

能够根据训练/验证曲线识别过拟合，并区分 Dropout、权重衰减、增强和早停的作用。


## 概念模型与执行路径

正则化不是单一 API：数据增强改变输入分布，权重衰减限制参数规模，Dropout 随机屏蔽激活，早停根据验证集选择训练时刻。


### 实验 1：定位课程正则化组件

**实验目的**：定位包含 `common` 包的课程根目录，为后续导入共享图像模型做准备。候选路径覆盖从仓库根目录、相邻目录和课程目录启动 Jupyter 的常见情况；插入前先检查可避免重复修改 `sys.path`。

若没有候选路径包含 `common`，`next(...)` 会抛出 `StopIteration`，应检查工作目录而不是继续排查模型。正式工程更适合安装包或使用固定入口。

In [ ]:
from pathlib import Path
import sys

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "07-deep-learning/pytorch",
]
PYTORCH_ROOT = next(path for path in candidates if (path / "common").exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))
print("course root:", PYTORCH_ROOT)


### 实验 2：在分类头中加入 Dropout

**实验目的**：比较相同 CNN 在 `dropout=0.0` 和 `dropout=0.5` 下的结构差异。两者卷积特征提取器相同，只有分类头 `Flatten → Dropout → Linear` 中的丢弃概率不同。

Dropout 没有可训练参数；训练时每个前向过程独立地以概率 0.5 将激活置零，并把保留激活除以 0.5，使期望值保持不变。它作用于展平后的 512 维特征，而不是直接缩小参数数量。打印分类头应看到 `Dropout(p=0.5)`。

**边界**：Dropout 通过增加训练噪声抑制特征之间的脆弱共适应，但概率过高会造成欠拟合。是否有效必须比较验证表现，不能仅凭训练 loss 判断。

In [ ]:
import torch
from common.models import ImageClassifier
plain = ImageClassifier(dropout=0.0)
regularized = ImageClassifier(dropout=0.5)
print(regularized.classifier)


### 实验 3：验证 Dropout 的训练/评估模式

**实验目的**：对同一输入重复前向，直接观察 Dropout 的模式依赖行为。`train()` 下两次随机 mask 通常不同，所以输出不相等；`eval()` 下 Dropout 成为恒等映射，两次结果应相等。

`train()`/`eval()` 只切换模块的 `training` 标志，不控制 Autograd。这里即使在 eval 模式，输出仍可能连接计算图；真实验证应同时使用 `torch.inference_mode()`。`torch.allclose` 比逐位相等更适合浮点结果。

**预期输出**：`train outputs equal: False`，`eval outputs equal: True`。极端随机巧合理论上存在，但对当前网络几乎可以忽略。

In [ ]:
sample = torch.randn(4, 1, 28, 28)
regularized.train()
first = regularized(sample)
second = regularized(sample)
regularized.eval()
third = regularized(sample)
fourth = regularized(sample)
print("train outputs equal:", torch.allclose(first, second))
print("eval outputs equal:", torch.allclose(third, fourth))


### 实验 4：配置 AdamW 的解耦权重衰减

**实验目的**：确认优化器参数组记录了 `weight_decay=1e-4`。AdamW 将参数衰减作为与自适应梯度更新解耦的步骤，直观上每次把参数向 0 收缩一点，从而限制过大的权重。

它与把 $\lambda\|w\|_2^2$ 直接加到 loss、再让 Adam 自适应缩放该梯度并不完全等价。实际项目常把 bias 和归一化层参数放入 weight decay 为 0 的独立参数组；本例为了简洁，对所有参数使用同一衰减。

**观察重点**：打印配置只能证明超参数已写入 optimizer，不能证明泛化一定改善；必须通过受控实验比较。

In [ ]:
optimizer = torch.optim.AdamW(regularized.parameters(), lr=1e-3, weight_decay=1e-4)
print("weight decay:", optimizer.param_groups[0]["weight_decay"])


### 实验 5：从训练/验证损失识别过拟合

**实验目的**：用人工构造的曲线练习区分优化进展和泛化退化。训练 loss 持续下降，说明模型仍在更好地拟合训练集；验证 loss 在索引 3 附近达到最低后上升，说明继续训练开始过拟合。

虚线标记索引 3，即列表中的第 4 个观测点。若横轴代表从 1 开始的 epoch，展示时应显式传入 epoch 序列或把标注改成对应编号，避免 off-by-one。最佳 checkpoint 应依据预先确定的验证指标保存；若监控 validation loss，则此处保存最低点。

**结果解读**：这不是优化失败，因为训练目标仍在改善；它是训练分布与未见数据表现分离的泛化失败。早停不会让模型更简单，而是选择训练轨迹上的合适时刻。

In [ ]:
import matplotlib.pyplot as plt
train_loss = [1.0, .7, .45, .25, .12, .06]
validation_loss = [1.1, .75, .5, .42, .55, .8]
plt.plot(train_loss, label="train")
plt.plot(validation_loss, label="validation")
plt.axvline(3, color="black", linestyle="--", label="best checkpoint")
plt.legend(); plt.show()


## 底层机制

正则化作用在不同边界：数据增强改变训练输入分布；Dropout 扰动中间表示；weight decay 改变参数更新；早停选择训练时刻。它们可能互补，也可能叠加过强导致训练和验证都表现差。

早停的 patience 用来容忍验证指标的短期噪声。每次改善时应保存 checkpoint；触发停止后重新加载最佳权重。测试集不能参与 patience、强度或 checkpoint 的选择，否则测试指标会带有选择偏差。

## 检查点

回答并验证：1）训练 loss 降、验证 loss 升属于优化失败还是泛化失败？2）`eval()` 是否禁用梯度？3）Dropout 为什么训练时缩放保留激活？4）AdamW weight decay 与 Adam+L2 为何不完全等价？5）图中虚线索引 3 对应第几个观测点？6）应依据验证集还是测试集保存最佳轮次？

## 试一试

固定划分、种子和训练预算，分别关闭增强、Dropout 和 weight decay，记录训练/验证 loss、accuracy 与最佳 epoch；再尝试 `dropout=0.8` 观察欠拟合。为 bias/归一化参数创建无衰减参数组，并比较参数范数。最后实现基于 validation loss 的早停，验证停止后加载的是最低 loss checkpoint。

## 常见错误与调试

- **用测试集调早停或正则化**：造成选择泄漏；只用验证集调参。
- **验证忘记 `eval()`**：Dropout 随机、指标波动；评估前切换模式。
- **以为 `eval()` 会禁用梯度**：仍会构图；配合 `inference_mode()`。
- **正则化过强**：训练和验证都差；降低 dropout、衰减或增强强度。
- **只看 accuracy**：置信度恶化可能先反映在 loss；同时追踪两者。
- **未保存最佳权重**：早停只停止循环；改善时保存并在测试前恢复。